In [5]:
%%writefile ml_pipeline.py

# ============================================================
# UNIT-2 - PRACTICE
# END-TO-END ML PIPELINE IMPLEMENTATION
# KIDNEY DISEASE CLASSIFICATION
# ============================================================

from pathlib import Path
import json
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = Path(__file__).resolve().parent

# Kidney Disease dataset
DATA_PATH = BASE_DIR / "kidney_disease_cleaned.csv"

# Output folders
ARTIFACT_DIR = BASE_DIR / "artifacts"
OUTPUT_DIR = BASE_DIR / "outputs"

# Saved complete ML pipeline
MODEL_PATH = ARTIFACT_DIR / "kidney_disease_pipeline.joblib"

# Output files
REPORT_PATH = OUTPUT_DIR / "pipeline_report.txt"
METRICS_PATH = OUTPUT_DIR / "metrics.json"
PREDICTION_PATH = OUTPUT_DIR / "sample_prediction.csv"


# ============================================================
# STEP 1 - LOAD DATA
# ============================================================

def load_data():

    print("\nSTEP 1: Loading Kidney Disease dataset...")

    # Load CSV file
    df = pd.read_csv(DATA_PATH)

    # Remove duplicate records
    df = df.drop_duplicates()

    print("Dataset loaded successfully")
    print("Dataset shape:", df.shape)

    print("\nFirst 5 records:")
    print(df.head())

    return df


# ============================================================
# STEP 2 - PREPARE DATA
# ============================================================

def prepare_data(df):

    print("\nSTEP 2: Preparing data...")

    # --------------------------------------------------------
    # Target column
    # --------------------------------------------------------

    y = df["classification"]

    # --------------------------------------------------------
    # Input features
    # --------------------------------------------------------

    X = df.drop("classification", axis=1)

    # --------------------------------------------------------
    # Remove ID column
    # ID is an identifier and should not be used as a feature.
    # --------------------------------------------------------

    if "id" in X.columns:
        X = X.drop("id", axis=1)

    # --------------------------------------------------------
    # Identify numerical columns automatically
    # --------------------------------------------------------

    num_cols = X.select_dtypes(
        include=["int64", "float64"]
    ).columns.tolist()

    # --------------------------------------------------------
    # Identify categorical columns automatically
    # --------------------------------------------------------

    cat_cols = X.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    print("\nNumber of input features:", X.shape[1])

    print("\nNumerical columns:")
    print(num_cols)

    print("\nCategorical columns:")
    print(cat_cols)

    return X, y, num_cols, cat_cols


# ============================================================
# STEP 3 - SPLIT DATA
# ============================================================

def split_data(X, y):

    print("\nSTEP 3: Splitting dataset...")

    # 80% training
    # 20% testing
    # random_state=42 gives reproducible results
    # stratify=y maintains class distribution

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    print("Training records:", X_train.shape[0])
    print("Testing records :", X_test.shape[0])

    return X_train, X_test, y_train, y_test


# ============================================================
# STEP 4 - BUILD PREPROCESSING + MODEL PIPELINE
# ============================================================

def build_pipeline(num_cols, cat_cols):

    print("\nSTEP 4: Building ML pipeline...")

    # --------------------------------------------------------
    # Numerical preprocessing
    #
    # Missing numerical values
    #        ↓
    # Median imputation
    #        ↓
    # Standard scaling
    # --------------------------------------------------------

    numeric_transformer = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    # --------------------------------------------------------
    # Categorical preprocessing
    #
    # Missing categorical values
    #        ↓
    # Most frequent value
    #        ↓
    # One-Hot Encoding
    # --------------------------------------------------------

    categorical_transformer = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ])

    # --------------------------------------------------------
    # Combine numerical and categorical preprocessing
    # --------------------------------------------------------

    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_transformer,
            num_cols
        ),
        (
            "cat",
            categorical_transformer,
            cat_cols
        )
    ])

    # --------------------------------------------------------
    # Machine Learning model
    # --------------------------------------------------------

    classifier = LogisticRegression(
        C=0.5,
        max_iter=2000,
        random_state=42
    )

    # --------------------------------------------------------
    # Complete end-to-end pipeline
    #
    # Raw Data
    #    ↓
    # Preprocessing
    #    ↓
    # Logistic Regression
    #    ↓
    # Prediction
    # --------------------------------------------------------

    pipeline = Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            classifier
        )
    ])

    print("Pipeline created successfully")

    return pipeline


# ============================================================
# STEP 5 - TRAIN MODEL
# ============================================================

def train_model(
    pipeline,
    X_train,
    y_train
):

    print("\nSTEP 5: Training Kidney Disease model...")

    pipeline.fit(
        X_train,
        y_train
    )

    print("Model training completed successfully")

    return pipeline


# ============================================================
# STEP 6 - EVALUATE MODEL
# ============================================================

def evaluate_model(
    pipeline,
    X_test,
    y_test
):

    print("\nSTEP 6: Evaluating model...")

    # Generate predictions
    y_pred = pipeline.predict(X_test)

    # --------------------------------------------------------
    # Calculate evaluation metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    # Store metrics
    metrics = {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1)
    }

    print("\nMODEL PERFORMANCE")
    print("-------------------------")

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            y_pred,
            zero_division=0
        )
    )

    return metrics, y_pred


# ============================================================
# STEP 7 - SAVE COMPLETE PIPELINE
# ============================================================

def save_model(pipeline):

    print("\nSTEP 7: Saving trained pipeline...")

    # Create artifacts directory
    ARTIFACT_DIR.mkdir(
        exist_ok=True
    )

    # Save complete pipeline
    joblib.dump(
        pipeline,
        MODEL_PATH
    )

    print("Pipeline saved successfully")
    print("Location:", MODEL_PATH)


# ============================================================
# STEP 8 - LOAD SAVED MODEL AND PREDICT
# ============================================================

def test_saved_model(X_test):

    print("\nSTEP 8: Loading saved pipeline...")

    # Load complete pipeline
    loaded_pipeline = joblib.load(
        MODEL_PATH
    )

    print("Saved pipeline loaded successfully")

    # --------------------------------------------------------
    # Select one patient from test data
    # --------------------------------------------------------

    sample = X_test.iloc[[0]].copy()

    # --------------------------------------------------------
    # Make prediction
    # --------------------------------------------------------

    prediction = loaded_pipeline.predict(
        sample
    )[0]

    # Convert NumPy value to Python value if necessary
    if hasattr(prediction, "item"):
        prediction = prediction.item()

    print("\nSample patient:")
    print(sample)

    print("\nPrediction:", prediction)

    # --------------------------------------------------------
    # Save prediction
    # --------------------------------------------------------

    result = sample.copy()

    result["Prediction"] = prediction

    OUTPUT_DIR.mkdir(
        exist_ok=True
    )

    result.to_csv(
        PREDICTION_PATH,
        index=False
    )

    print(
        "Prediction saved:",
        PREDICTION_PATH
    )

    return prediction


# ============================================================
# STEP 9 - SAVE METRICS AND REPORT
# ============================================================

def save_report(
    metrics,
    prediction
):

    print("\nSTEP 9: Saving execution results...")

    # Create output directory
    OUTPUT_DIR.mkdir(
        exist_ok=True
    )

    # --------------------------------------------------------
    # Save metrics as JSON
    # --------------------------------------------------------

    with open(
        METRICS_PATH,
        "w"
    ) as f:

        json.dump(
            metrics,
            f,
            indent=4
        )

    # --------------------------------------------------------
    # Save human-readable report
    # --------------------------------------------------------

    with open(
        REPORT_PATH,
        "w"
    ) as f:

        f.write(
            "UNIT-2 - END-TO-END ML PIPELINE\n"
        )

        f.write(
            "KIDNEY DISEASE CLASSIFICATION\n"
        )

        f.write(
            "=======================================\n\n"
        )

        f.write(
            "Dataset: kidney_disease_cleaned.csv\n"
        )

        f.write(
            "Model: Logistic Regression\n"
        )

        f.write(
            "Pipeline: Preprocessing + Training + Prediction\n\n"
        )

        f.write(
            f"Accuracy  : {metrics['accuracy']:.4f}\n"
        )

        f.write(
            f"Precision : {metrics['precision']:.4f}\n"
        )

        f.write(
            f"Recall    : {metrics['recall']:.4f}\n"
        )

        f.write(
            f"F1 Score  : {metrics['f1_score']:.4f}\n"
        )

        f.write(
            f"\nSample Prediction: {prediction}\n"
        )

        f.write(
            "\nPipeline executed successfully.\n"
        )

    print("Metrics saved :", METRICS_PATH)
    print("Report saved  :", REPORT_PATH)


# ============================================================
# MAIN FUNCTION
# ============================================================

def main():

    print("\n==========================================")
    print(" KIDNEY DISEASE ML PIPELINE")
    print("==========================================")

    # --------------------------------------------------------
    # STEP 1 - Load data
    # --------------------------------------------------------

    df = load_data()

    # --------------------------------------------------------
    # STEP 2 - Prepare data
    # --------------------------------------------------------

    X, y, num_cols, cat_cols = prepare_data(
        df
    )

    # --------------------------------------------------------
    # STEP 3 - Split data
    # --------------------------------------------------------

    X_train, X_test, y_train, y_test = split_data(
        X,
        y
    )

    # --------------------------------------------------------
    # STEP 4 - Build pipeline
    # --------------------------------------------------------

    pipeline = build_pipeline(
        num_cols,
        cat_cols
    )

    # --------------------------------------------------------
    # STEP 5 - Train model
    # --------------------------------------------------------

    pipeline = train_model(
        pipeline,
        X_train,
        y_train
    )

    # --------------------------------------------------------
    # STEP 6 - Evaluate model
    # --------------------------------------------------------

    metrics, y_pred = evaluate_model(
        pipeline,
        X_test,
        y_test
    )

    # --------------------------------------------------------
    # STEP 7 - Save model
    # --------------------------------------------------------

    save_model(
        pipeline
    )

    # --------------------------------------------------------
    # STEP 8 - Reload model and predict
    # --------------------------------------------------------

    prediction = test_saved_model(
        X_test
    )

    # --------------------------------------------------------
    # STEP 9 - Save outputs
    # --------------------------------------------------------

    save_report(
        metrics,
        prediction
    )

    print("\n==========================================")
    print(" PIPELINE COMPLETED SUCCESSFULLY")
    print("==========================================")

    print("\nGenerated files:")

    print("1.", MODEL_PATH)
    print("2.", METRICS_PATH)
    print("3.", REPORT_PATH)
    print("4.", PREDICTION_PATH)


# ============================================================
# START EXECUTION
# ============================================================

if __name__ == "__main__":
    main()

Overwriting ml_pipeline.py


In [6]:
import os

print(
    "ml_pipeline.py exists:",
    os.path.exists("ml_pipeline.py")
)

print(
    "Current folder:",
    os.getcwd()
)

ml_pipeline.py exists: True
Current folder: C:\Users\Admin
